# 02 · Fault Discretization and the Green's Function Matrix

The forward model $\mathbf{d} = G\mathbf{m}$ hides all of its physics inside the
matrix $G$. To invert real data well — and to understand *why* inversions
struggle — you need to know what $G$ actually contains. This notebook builds $G$
from the ground up.

## Learning objectives
- See how a continuous fault is approximated by discrete patches.
- Recognize $G$ as a **design matrix**: each column is one basis response.
- Build $G$ column by column by hand, then reproduce it with one `geodef` call.
- Read the **blocked** column layout (`[strike-slip | dip-slip]`) and the
  interleaved row layout (`[e, n, u, ...]`).
- Use $G$ for forward prediction with a multi-patch slip model.

## The design matrix idea

Any *linear* model can be written $\mathbf{d} = G\mathbf{m}$, where each column
of $G$ says how one model parameter contributes to the data. For this reason $G$
is often called the **design matrix**.

The simplest example is fitting a straight line $y = a x + b$. Stacking all
observations,

$$ \mathbf{y} = G\mathbf{m},\qquad
G = \begin{bmatrix} x_1 & 1\\ x_2 & 1\\ \vdots & \vdots\\ x_M & 1\end{bmatrix},\qquad
\mathbf{m} = \begin{bmatrix} a\\ b\end{bmatrix}. $$

Column 1 (the $x_i$) scales with the slope; column 2 (all ones) scales with the
intercept. Fault-slip modeling is the *same idea* — only the columns are elastic
displacement fields instead of `x` and `1`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(7)

# warm-up: the simplest design matrix (line fit)
x = np.linspace(0, 10, 30)
y = 1.8 * x - 0.7 + rng.normal(0, 0.8, x.size)

G_line = np.column_stack([x, np.ones_like(x)])     # (M, 2) design matrix
m_line, *_ = np.linalg.lstsq(G_line, y, rcond=None)
print(f"Recovered slope = {m_line[0]:.2f}, intercept = {m_line[1]:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, s=18, label="data")
ax.plot(x, G_line @ m_line, "r", label="least-squares fit")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
ax.set_title("A two-column design matrix")
plt.show()

Recovered slope = 1.75, intercept = -0.76


/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99071/1145332437.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## From lines to faults

Now replace the two columns of the line-fit example with *hundreds* of columns —
one per fault-patch slip component. Each column is the **surface displacement
field produced by one unit (1 m) of slip on a single patch**, holding every
other patch at zero. Scaled by the actual slip amounts and summed, these basis
fields reproduce the full deformation:

$$ \mathbf{d} = \sum_{k} m_k\, \underbrace{G_{:,k}}_{\text{patch } k \text{ unit response}} = G\mathbf{m}. $$

Let's build a small fault so the matrix stays easy to inspect.

In [2]:
fault = geodef.Fault.planar(
    lat=0.0, lon=100.0, depth=10e3,
    strike=315.0, dip=20.0,
    length=40e3, width=16e3,
    n_length=8, n_width=4,            # 32 patches -> small, inspectable G
)
N = fault.n_patches
nL, nW = fault.grid_shape

# surface station grid
ox, oy = np.meshgrid(np.linspace(99.6, 100.4, 9), np.linspace(-0.4, 0.4, 9))
obs_lon, obs_lat = ox.ravel(), oy.ravel()
M = obs_lat.size
print(f"{N} patches ({nL} x {nW}),  {M} stations  ->  G is ({3 * M}, {2 * N})")

32 patches (8 x 4),  81 stations  ->  G is (243, 64)


## Build $G$, column by column

### By hand

Column $k$ of $G$ is just the forward model run with **unit slip on patch $k$
only**. We loop over patches, switch on one unit of slip at a time, and drop the
resulting displacement field into the matching column. The first $N$ columns are
strike-slip; the next $N$ are dip-slip.

In [3]:
G_manual = np.zeros((3 * M, 2 * N))

for k in range(N):                       # strike-slip columns  [0 : N]
    unit = np.zeros(N); unit[k] = 1.0
    ue, un, uz = fault.displacement(obs_lat, obs_lon,
                                    slip_strike=unit, slip_dip=0.0)
    G_manual[0::3, k], G_manual[1::3, k], G_manual[2::3, k] = ue, un, uz

for k in range(N):                       # dip-slip columns  [N : 2N]
    unit = np.zeros(N); unit[k] = 1.0
    ue, un, uz = fault.displacement(obs_lat, obs_lon,
                                    slip_strike=0.0, slip_dip=unit)
    G_manual[0::3, N + k] = ue
    G_manual[1::3, N + k] = un
    G_manual[2::3, N + k] = uz

print("Built G_manual:", G_manual.shape)

Built G_manual: (243, 64)


### With geodef

`fault.greens_matrix()` assembles the very same matrix in a single vectorized
call — no Python loop over patches. When you build $G$ for a *dataset*,
`geodef.greens.greens(fault, dataset)` does this and then projects each column
into the data's observation space. For 3-component GNSS that projection is just
the `[e, n, u]` interleaving, so the two agree exactly.

In [4]:
G = fault.greens_matrix(obs_lat, obs_lon)
print("greens_matrix == hand-built:", np.allclose(G, G_manual))

# the dataset-aware assembler gives the same matrix for 3-component GNSS
gnss = geodef.GNSS(obs_lon, obs_lat,
                   np.zeros(M), np.zeros(M), np.zeros(M),
                   np.ones(M), np.ones(M), np.ones(M))
G_proj = geodef.greens.greens(fault, gnss)
print("greens(fault, gnss) == greens_matrix:", np.allclose(G_proj, G))

greens_matrix == hand-built: True
greens(fault, gnss) == greens_matrix: True


## The structure of $G$

- **Columns are blocked:** `G[:, :N]` are the strike-slip responses, `G[:, N:]`
  the dip-slip responses.
- **Rows are interleaved by station:** `[e₁, n₁, u₁, e₂, n₂, u₂, …]`.

Visualizing the whole matrix makes the two column blocks obvious. The vertical
line separates strike-slip (left) from dip-slip (right).

In [5]:
fig, ax = plt.subplots(figsize=(8, 5))
vmax = np.abs(G).max()
im = ax.imshow(G, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.axvline(N - 0.5, color="k", lw=1.2)
ax.set_xlabel("model parameter (column):  strike-slip  |  dip-slip")
ax.set_ylabel("observation (row):  [e, n, u] per station")
ax.set_title("The Green's matrix G")
fig.colorbar(im, ax=ax, label="displacement per unit slip (m / m)")
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99071/4251835502.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### One column is one patch's "footprint"

Reading a single column back out as a displacement field shows what it really
is: the surface motion from 1 m of slip on one patch. Here is the dip-slip
column for a patch in the middle of the fault, drawn over the highlighted patch.

In [6]:
k = fault.patch_index(strike_idx=4, dip_idx=1)
col = G[:, N + k]                        # dip-slip column for patch k
ce, cn, cu = col[0::3], col[1::3], col[2::3]

resp = geodef.GNSS(obs_lon, obs_lat, ce, cn, cu,
                   np.ones(M) * 1e-6, np.ones(M) * 1e-6, np.ones(M) * 1e-6)

fig, ax = plt.subplots(figsize=(6.5, 6))
highlight = np.zeros(N); highlight[k] = 1.0
geodef.plot.slip(fault, highlight, ax=ax, cmap="Reds", colorbar=False)

scale = 12.0 / np.hypot(ce, cn).max()
geodef.plot.vectors(resp, fault, ax=ax, components="horizontal", scale=scale,
                    title=f"Column {N + k}: unit dip-slip on patch {k}")
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99071/920193740.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Forward prediction with many patches

With $G$ in hand, predicting data for *any* slip model is one matrix multiply.
Let's build a slip distribution with two asperities and predict the surface
displacements — note how little code this takes now that the physics lives in
$G$.

In [7]:
i = np.arange(N) % nL
j = np.arange(N) // nL
slip = (1.0 * np.exp(-(((i - 2) / 1.2) ** 2 + ((j - 1) / 1.0) ** 2))
        + 0.7 * np.exp(-(((i - 6) / 1.0) ** 2 + ((j - 2) / 0.8) ** 2)))

m = np.concatenate([np.zeros(N), slip])      # dip-slip only
d = G @ m
pe, pn, pu = d[0::3], d[1::3], d[2::3]

pred = geodef.GNSS(obs_lon, obs_lat, pe, pn, pu,
                   np.ones(M) * 1e-3, np.ones(M) * 1e-3, np.ones(M) * 1e-3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
geodef.plot.slip(fault, slip, ax=axes[0], cmap="YlOrRd",
                 colorbar_label="Dip-slip (m)",
                 title="Slip model (two asperities)")
scale = 12.0 / np.hypot(pe, pn).max()
geodef.plot.vectors(pred, fault, ax=axes[1], components="both", scale=scale,
                    title="Predicted displacements")
plt.show()

/var/folders/sk/yyr_g66j6rsdlxf5ss0mrqj40000gn/T/ipykernel_99071/2533803091.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Why discretization is a balancing act

More, smaller patches give $G$ more columns and let slip vary in finer detail —
but each column adds an unknown. Far from the stations, neighboring patches
produce almost identical surface displacements, so their columns become nearly
parallel. When that happens, many different slip models fit the data almost
equally well and the raw inverse problem becomes **ill-posed**. Notebook 03
demonstrates that instability directly, and notebook 04 introduces the
regularization used to tame it.

## Exercises
1. **Refine the grid.** Rebuild the fault with `n_length=16, n_width=8` and
   re-display $G$ with `imshow`. How do the blocks and the column count change?
2. **Strike vs dip.** Plot a strike-slip column (`G[:, k]`) for the same patch
   `k` and compare its vector pattern to the dip-slip column above.
3. **Column similarity.** Compare two *adjacent deep* patch columns with two
   *adjacent shallow* ones (e.g. with `np.corrcoef`). Which pair is more nearly
   parallel, and what does that imply for resolving slip at depth?

## Checkpoint questions
1. What are the dimensions of $G$ for $N$ patches and $M$ three-component
   stations, and why?
2. Why are the slip columns *blocked* `[strike | dip]` rather than interleaved?
3. Two columns of $G$ are nearly identical. What does that say about how well the
   data can distinguish slip on those two patches?

## Common mistakes
- Forgetting that `greens_matrix` returns **both** slip components ($2N$
  columns). Slice `[:, :N]` or `[:, N:]` if you want only one.
- Assuming finer patches always give a better answer — without regularization,
  they usually make the inversion *less* stable.
- Mixing up the interleaved row order: vertical is `d[2::3]`, not `d[2*M:]`.

## Summary
- $G$ is a design matrix: every column is one patch's unit-slip displacement
  field.
- `fault.greens_matrix()` (or `geodef.greens.greens()` for a dataset) builds it
  in one call; we verified it equals the hand-built version.
- Columns are blocked `[strike | dip]`; rows are interleaved `[e, n, u]`.
- Discretization trades resolution against stability — the motivation for the
  inversion and regularization notebooks that follow.